In [ ]:
# Create benchmark data

# In the first time, run first: 
# !chmod +x init.sh
# !./init.sh

# Prepare the base system's agents:
!./simple_run.sh core.system base

# Prepare agents for the second scenario: 
# !chmod +x prepare_data_system2.sh
# !./prepare_data_system2.sh


# Prepare agents for the third scenario: 
# !chmod +x prepare_data_system3.sh
# !./prepare_data_system3.sh

In [ ]:
# Process the data
from process_csvs import process_csv_files
experiment_name = "base"

process_csv_files(f"{experiment_name}/results/train", f"{experiment_name}/results/train/output")
process_csv_files(f"{experiment_name}/results/test", f"{experiment_name}/results/test/output")

from merge import merge_agent_scores
merge_agent_scores(f"{experiment_name}/results/train/output", f"{experiment_name}/train.csv")
merge_agent_scores(f"{experiment_name}/results/test/output", f"{experiment_name}/test.csv")

In [11]:
#### uncomment only the test you want to run

# baseline 
# !python main.py --train-data base/train.csv --test-data base/test.csv --embedding base/embedding.pt --system core.system --output base/output.csv
!python main.py --train-data base_all_claude_official/train.csv --test-data base_all_claude_official/test.csv --embedding base_all_claude_official/embedding.pt --system experiments.system_base_all_claude_official --output base_all_claude_official/output.csv
# !python main.py --train-data base_all_kimi25_official/train.csv --test-data base_all_kimi25_official/test.csv --embedding base_all_kimi25_official/embedding.pt --system experiments.system_base_all_kimi25_official --output base_all_kimi25_official/output.csv

# short description
# !python main.py --train-data short_description/train.csv --test-data short_description/test.csv --embedding short_description/embedding.pt --system experiments.system_short_description --output short_description/output.csv

# long description
# !python main.py --train-data long_description/train.csv --test-data long_description/test.csv --embedding long_description/embedding.pt --system experiments.system_long_description --output long_description/output.csv

# short system prompt
# !python main.py --train-data short_sysprompt/train.csv --test-data short_sysprompt/test.csv --embedding short_sysprompt/embedding.pt --system experiments.system_short_sysprompt --output short_sysprompt/output.csv

# long system prompt
# !python main.py --train-data long_sysprompt/train.csv --test-data long_sysprompt/test.csv --embedding long_sysprompt/embedding.pt --system experiments.system_long_sysprompt --output long_sysprompt/output.csv

# very long system prompt
# !python main.py --train-data long_sysprompt_1000/train.csv --test-data long_sysprompt_1000/test.csv --embedding long_sysprompt_1000/embedding.pt --system experiments.system_long_sysprompt_1000 --output long_sysprompt_1000/output.csv

# second scenario
# !python main.py --train-data scenario2/train.csv --test-data scenario2/test.csv --embedding scenario2/embedding.pt --system experiments.system_scenario2 --output scenario2/output.csv

# third scenario
# !python main.py --train-data scenario3/train.csv --test-data scenario3/test.csv --embedding scenario3/embedding.pt --system experiments.system_scenario3 --output scenario3/output.csv

# fixed model = llama3.2
# !python main.py --train-data fixed_model/train.csv --test-data fixed_model/test.csv --embedding fixed_model/embedding.pt --system experiments.fixed_model --output fixed_model/output.csv

# fixed model = qwen2.5
# !python main.py --train-data fixed_model_qwen/train.csv --test-data fixed_model_qwen/test.csv --embedding fixed_model_qwen/embedding.pt --system experiments.fixed_model_qwen --output fixed_model_qwen/output.csv

# 2 agents 
# !python main.py --train-data num_agents_2/train.csv --test-data num_agents_2/test.csv --embedding num_agents_2/embedding.pt --system experiments.num_agents_2 --output num_agents_2/output.csv

# 20 agents 
# !python main.py --train-data num_agents_20/train.csv --test-data num_agents_20/test.csv --embedding num_agents_20/embedding.pt --system experiments.num_agents_20 --output num_agents_20/output.csv

--- [Main] Loading System Module: experiments.system_base_all_claude_official ---
--- [Health Check] Pinging LLM Server at http://127.0.0.1:11434/v1... ---
✅ [Health Check] Server is UP!
--- [Router] Using device: cuda ---
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 21460.58it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
--- [Router] Loading Embedding Matrix from base_all_claude_official/embedding.pt ---
/workspace/ANTAP/core/router.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See htt

Process the data against the Malicious agent tool calls to get ASR

In [17]:
from analyzer import calculate_comparison
experiment = "base_all_kimi25_official"
calculate_comparison(malicious_path=f"{experiment}/results/test/Malicious.csv", benchmark_path=f"{experiment}/output.csv")

--- Security Prompts (n=0) ---
ANTAP   | ASR: 0.00% | Performance: 0.00%
AutoGen | ASR: 0.00% | Performance: 0.00%

--- Overall Dataset (n=0) ---
ANTAP   | ASR: 0.00% | Performance: 0.00%
AutoGen | ASR: 0.00% | Performance: 0.00%


In [25]:
import pandas as pd

def analyze_benchmark(file_path):
    df = pd.read_csv(file_path)
    
    total_prompts = len(df)
    
    antap_asr = (df['antap_score'].sum() / total_prompts) * 100
    antap_avg_time = df['antap_time_sec'].mean()
    
    autogen_asr = (df['autogen_score'].sum() / total_prompts) * 100
    autogen_avg_time = df['autogen_time_sec'].mean()
    
    print("=== ANTAP System ===")
    print(f"ASR: {antap_asr:.2f}%")
    print(f"Average Runtime: {antap_avg_time:.4f} seconds\n")
    
    print("=== AutoGen System ===")
    print(f"ASR: {autogen_asr:.2f}%")
    print(f"Average Runtime: {autogen_avg_time:.4f} seconds")

# Run the analysis on your specific file
if __name__ == "__main__":
    analyze_benchmark("base_all_claude_official/agentharm_benchmark.csv")

=== ANTAP System ===
ASR: 50.00%
Average Runtime: 0.0043 seconds

=== AutoGen System ===
ASR: 29.55%
Average Runtime: 0.2121 seconds
